In [669]:
import os
import pandas as pd
import numpy as np
import math

In [670]:
from pathlib import Path

# Get the current notebook's directory
CURRENT_NOTEBOOK_DIR = Path().resolve()

# Assume project root is one level up from notebooks/
BASE_DIR = CURRENT_NOTEBOOK_DIR.parent.parent
Processed_Data_Path = BASE_DIR / "data"/ "processed"/ "cleaned"
Processed_Data_Path

WindowsPath('D:/Repos/Data_viz/technerds/data/processed/cleaned')

In [671]:
applications = pd.read_csv(Processed_Data_Path/"application_data.csv")
applications.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [672]:
applications.shape

(36102, 122)

In [673]:
applications.isna().sum().sum()

np.int64(0)

## Step:0 Basic Cleaning  
Remove ID (not useful for EDA)

In [674]:
applications.drop(columns=['SK_ID_CURR'], inplace=True, errors='ignore')

# Step 1: Convert DAYS variables from negative days → positive years

Several columns in the dataset store information as **negative day counts**, indicating how many days *before* the application a certain event took place. These columns are:

- `DAYS_BIRTH`
- `DAYS_EMPLOYED`
- `DAYS_REGISTRATION`
- `DAYS_ID_PUBLISH`
- `DAYS_LAST_PHONE_CHANGE`

### What We Did

We transformed these negative day values into **positive year values** by:

- Dividing each value by **365** to convert days into years  
- Multiplying by **-1** to make the values positive  
- Rounding and converting them to **integers**  
- Creating new, interpretable features:
  - `AGE_AT_LOAN`
  - `JOB_AGE_AT_LOAN`
  - `DAYS_REGISTRATION`
  - `DOCS_CHANGE_AT`
  - `YEARS_LAST_PHONE_CHANGE`
- Dropping the original negative-day columns to avoid redundancy

### Why We Did It

- **Improves interpretability:** Years are easier to understand and work with than negative day counts  
- **Better for machine learning:** More meaningful features help models learn clearer relationships  
- **Removes confusing negative values:** Avoids misinterpretation during EDA or model building  
- **Reduces noise:** Prevents having multiple representations of the same information  

In [675]:
def convert_to_positive_year(applications):
    applications['AGE_AT_LOAN'] = -round(applications['DAYS_BIRTH']/365).astype(int)
    applications['JOB_AGE_AT_LOAN'] = -round(applications['DAYS_EMPLOYED']/365).astype(int)
    applications['DAYS_REGISTRATION'] = -round(applications['DAYS_REGISTRATION']/365).astype(int)
    applications['DOCS_CHANGE_AT'] = -round(applications['DAYS_ID_PUBLISH']/365).astype(int)
    applications['YEARS_LAST_PHONE_CHANGE'] = -round(applications['DAYS_LAST_PHONE_CHANGE']/365).astype(int)
    cols_to_drop = [
        'DAYS_BIRTH',
        'DAYS_EMPLOYED',
        'DAYS_REGISTRATION',
        'DAYS_ID_PUBLISH',
        'DAYS_LAST_PHONE_CHANGE'
    ]
    # Drop them safely if present
    applications.drop(columns=[c for c in cols_to_drop if c in applications.columns], inplace=True)
convert_to_positive_year(applications)

# Step 2: Financial Ratios

To better capture an applicant’s financial stability, we engineered several **domain-specific ratios** using their income, credit, annuity, and family information. These ratios are extremely valuable in credit-risk modeling because they directly express **financial stress**, **repayment capacity**, and **borrowing behavior**.

### What We Did

We created the following new financial features:

---

### **1. Credit / Income Ratio**

**Formula:**  
`CREDIT_INCOME_RATIO = AMT_CREDIT / AMT_INCOME_TOTAL`

**Meaning:**  
How many years of income would be needed to repay the loan?

**Why it’s powerful:**  
- A higher ratio means the applicant is taking a loan far larger than their income.  
- Indicates **financial stress**, which strongly correlates with **default risk**.

---

### **2. Annuity / Income Ratio**

**Formula:**  
`ANNUITY_INCOME_RATIO = AMT_ANNUITY / AMT_INCOME_TOTAL`

**Meaning:**  
How large the monthly installment is relative to total income.

**Why it’s powerful:**  
- If the installment is too high compared to income, repayment becomes difficult.  
- One of the strongest predictors in credit-scoring models globally.

---

### **3. Credit / Goods Price Ratio**

**Formula:**  
`CREDIT_GOODS_RATIO = AMT_CREDIT / AMT_GOODS_PRICE`

**Meaning:**  
How much credit was taken compared to the value of the purchased item.

**Why it’s powerful:**  
- If this ratio is **> 1**, the borrower is taking more loan than the asset value.  
- Can indicate **over-borrowing** or **inflated credit line**, which increases risk.

---

### **4. Income per Family Member**

**Formula:**  
`INCOME_PER_PERSON = AMT_INCOME_TOTAL / CNT_FAM_MEMBERS`

**Meaning:**  
How much income is available per household member.

**Why it’s powerful:**  
- A large family with low income means higher living expenses.  
- Lower income per person = **higher probability of default**.

---

### Columns Removed

After creating the new ratios, the original base columns were removed to eliminate redundancy:

- `AMT_CREDIT`  
- `AMT_INCOME_TOTAL`  
- `AMT_ANNUITY`  
- `AMT_GOODS_PRICE`  
- `CNT_FAM_MEMBERS`

In [676]:
def Financial_Ratio(df):
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['CREDIT_GOODS_RATIO'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
    cols_to_drop = [
        'AMT_CREDIT',
        'AMT_INCOME_TOTAL',
        'AMT_ANNUITY',
        'AMT_GOODS_PRICE',
        'CNT_FAM_MEMBERS'
    ]
    # Drop only if they exist in the dataframe
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
Financial_Ratio(applications)

In [677]:
applications.shape

(36102, 119)

# **Step 3: Feature Aggregation**

In this step, multiple related columns were combined into single, more meaningful features.  
This reduces noise, removes redundancy, and creates stronger analytical signals.

---

## **1. TOTAL_CONTACT_POINTS (6 features → 1)**

**Sources Used:**  
- FLAG_MOBIL  
- FLAG_EMP_PHONE  
- FLAG_WORK_PHONE  
- FLAG_CONT_MOBILE  
- FLAG_PHONE  
- FLAG_EMAIL  

**Meaning:**  
Counts how many valid communication channels the applicant has provided.

**Why Important?**  
More contact points → applicant is easier to reach → lower operational risk.  
This feature captures customer accessibility in a single numeric value.

---

## **2. COUNT_DOCS (20+ document flags → 1)**

**Sources Used:**  
All `FLAG_DOCUMENT_*` columns  
(FLAG_DOCUMENT_2 through FLAG_DOCUMENT_21)

**Meaning:**  
How many official documents the applicant submitted.

**Why Important?**  
Submitting more documents indicates:  
- higher transparency  
- lower fraud intent  
- better verification quality  

This single feature captures “documentation completeness” efficiently.

---

## **3. EXT_SOURCE_MEAN (3 external scores → 1)**

**Sources Used:**  
- EXT_SOURCE_1  
- EXT_SOURCE_2  
- EXT_SOURCE_3  

These are external creditworthiness scores, each ranging from **0 to 1**.

**Meaning:**  
We compute the average of all available external risk scores.

**Why Important?**  
These scores are among the *strongest predictors* of default.  
Taking the mean stabilizes missing values and reduces noise while preserving the signal.

---

## **4. TOTAL_OBS_SOCIAL, TOTAL_DEF_SOCIAL & SOCIAL_RISK_RATIO**

**Sources Used (Observed):**  
- OBS_30_CNT_SOCIAL_CIRCLE  
- OBS_60_CNT_SOCIAL_CIRCLE  

**Sources Used (Defaults):**  
- DEF_30_CNT_SOCIAL_CIRCLE  
- DEF_60_CNT_SOCIAL_CIRCLE  

**New Features:**  
- `TOTAL_OBS_SOCIAL` = total observations  
- `TOTAL_DEF_SOCIAL` = total defaults  
- `SOCIAL_RISK_RATIO` = TOTAL_DEF_SOCIAL / (TOTAL_OBS_SOCIAL + 1)

**Meaning:**  
Represents the financial health of the applicant’s social circle.

**Why Important?**  
People heavily correlate with the spending and repayment behavior of those around them.  
A high default ratio in their social environment increases applicant risk.

---

## **5. TOTAL_BUREAU_REQUESTS (6 bureau inquiries → 1)**

**Sources Used:**  
All AMT_REQ_CREDIT_BUREAU_* columns  
(Hour, Day, Week, Month, Quarter, Year)

**Meaning:**  
Counts the number of times the applicant's profile was checked by the credit bureau.

**Why Important?**  
More inquiries → higher credit desperation → higher probability of default.

---

## **6. PROPERTY_AREA (40+ property attributes → 1)**

**Sources Used:**  
All *_AVG, *_MODE, *_MEDI property-related columns  
(e.g., APARTMENTS_AVG, FLOORSMAX_MODE, LIVINGAREA_MEDI, etc.)

**Meaning:**  
Single score representing the overall property size and quality.

**Why Important?**  
Rather than analyzing dozens of correlated property variables,  
this aggregated score captures the essence of living condition and wealth.

---

## ✔ Summary of Step 3
| Category | Original Columns | New Feature |
|---------|------------------|-------------|
| Contact Info | 6 | TOTAL_CONTACT_POINTS |
| Documents | 20+ | COUNT_DOCS |
| External Scores | 3 | EXT_SOURCE_MEAN |
| Social Behavior | 4 | SOCIAL_RISK_RATIO |
| Bureau History | 6 | TOTAL_BUREAU_REQUESTS |
| Property Characteristics | 40+ | PROPERTY_AREA |

👉 **We reduced 75+ columns to just 6 powerful features while improving the clarity and quality of the dataset.**

---




In [678]:
def Total_contact_point(df):
    contact_cols = [
        'FLAG_MOBIL','FLAG_EMP_PHONE','FLAG_WORK_PHONE',
        'FLAG_CONT_MOBILE','FLAG_PHONE','FLAG_EMAIL'
    ]
    existing_contact_cols = [c for c in contact_cols if c in df.columns]
    
    df['TOTAL_CONTACT_POINTS'] = df[existing_contact_cols].sum(axis=1)
    df.drop(columns=existing_contact_cols, inplace=True, errors='ignore')


def Count_doc_submitted(df):
    doc_cols = df.filter(like='FLAG_DOCUMENT').columns.tolist()
    
    df['COUNT_DOCS'] = df[doc_cols].sum(axis=1)
    df.drop(columns=doc_cols, inplace=True, errors='ignore')


def Ext_Source(df):
    ext_cols = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
    df['EXT_SOURCE_MEAN'] = df[ext_cols].mean(axis=1)
    df.drop(columns=ext_cols, inplace=True, errors='ignore')


def Social_circle(df):
    obs_cols = ['OBS_30_CNT_SOCIAL_CIRCLE','OBS_60_CNT_SOCIAL_CIRCLE']
    def_cols = ['DEF_30_CNT_SOCIAL_CIRCLE','DEF_60_CNT_SOCIAL_CIRCLE']
    
    df['TOTAL_OBS_SOCIAL']  = df[obs_cols].sum(axis=1)
    df['TOTAL_DEF_SOCIAL']  = df[def_cols].sum(axis=1)
    df['SOCIAL_RISK_RATIO'] = df['TOTAL_DEF_SOCIAL'] / (df['TOTAL_OBS_SOCIAL'] + 1)
    
    # Drop original + intermediate columns
    df.drop(columns=(obs_cols + def_cols + ['TOTAL_OBS_SOCIAL','TOTAL_DEF_SOCIAL']), 
            inplace=True, errors='ignore')


def credit_bureau_risk(df):
    bureau_cols = [
        'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
        'AMT_REQ_CREDIT_BUREAU_WEEK','AMT_REQ_CREDIT_BUREAU_MON',
        'AMT_REQ_CREDIT_BUREAU_QRT','AMT_REQ_CREDIT_BUREAU_YEAR'
    ]
    df['TOTAL_BUREAU_REQUESTS'] = df[bureau_cols].sum(axis=1)
    df.drop(columns=bureau_cols, inplace=True, errors='ignore')


def property_area(df):
    area_cols = [
        'APARTMENTS_AVG','BASEMENTAREA_AVG','YEARS_BEGINEXPLUATATION_AVG',
        'YEARS_BUILD_AVG','COMMONAREA_AVG','ELEVATORS_AVG','ENTRANCES_AVG',
        'FLOORSMAX_AVG','FLOORSMIN_AVG','LANDAREA_AVG','LIVINGAPARTMENTS_AVG',
        'LIVINGAREA_AVG','NONLIVINGAPARTMENTS_AVG','NONLIVINGAREA_AVG',
        'APARTMENTS_MODE','BASEMENTAREA_MODE','YEARS_BEGINEXPLUATATION_MODE',
        'YEARS_BUILD_MODE','COMMONAREA_MODE','ELEVATORS_MODE','ENTRANCES_MODE',
        'FLOORSMAX_MODE','FLOORSMIN_MODE','LANDAREA_MODE','LIVINGAPARTMENTS_MODE',
        'LIVINGAREA_MODE','NONLIVINGAPARTMENTS_MODE','NONLIVINGAREA_MODE',
        'APARTMENTS_MEDI','BASEMENTAREA_MEDI','YEARS_BEGINEXPLUATATION_MEDI',
        'YEARS_BUILD_MEDI','COMMONAREA_MEDI','ELEVATORS_MEDI','ENTRANCES_MEDI',
        'FLOORSMAX_MEDI','FLOORSMIN_MEDI','LANDAREA_MEDI','LIVINGAPARTMENTS_MEDI',
        'LIVINGAREA_MEDI','NONLIVINGAPARTMENTS_MEDI','NONLIVINGAREA_MEDI',
        'TOTALAREA_MODE'
    ]
    
    df['PROPERTY_AREA'] = df[area_cols].mean(axis=1)
    df.drop(columns=area_cols, inplace=True, errors='ignore')


def categorical_simplification(df):
    df['EDU_SIMPLIFIED'] = df['NAME_EDUCATION_TYPE'].replace({
        'Higher education':'Higher', 'Incomplete higher':'Higher',
        'Secondary / secondary special':'Secondary', 'Lower secondary':'Low',
        'Academic degree':'Higher'
    })

    df['INCOME_SIMPLIFIED'] = df['NAME_INCOME_TYPE'].replace({
        'Working':'Working', 'Commercial associate':'Commercial',
        'State servant':'Govt', 'Pensioner':'Pension',
        'Unemployed':'No Income', 'Student':'No Income',
        'Maternity leave':'No Income', 'Businessman':'Business'
    })

    df['FAMILY_TYPE'] = df['NAME_FAMILY_STATUS'].replace({
        'Married':'Married', 'Single / not married':'Single',
        'Civil marriage':'Civil', 'Separated':'Separated', 'Widow':'Widow'
    })

    df.drop(columns=['NAME_EDUCATION_TYPE','NAME_INCOME_TYPE','NAME_FAMILY_STATUS'],
            inplace=True, errors='ignore')


def regional_risk_score(df):
    df['REGION_RATING_DIFF'] = df['REGION_RATING_CLIENT_W_CITY'] - df['REGION_RATING_CLIENT']
    df.drop(columns=['REGION_RATING_CLIENT'], inplace=True, errors='ignore')


def mobility_score(df):
    mobility_cols = [
        'REG_REGION_NOT_LIVE_REGION','REG_REGION_NOT_WORK_REGION',
        'LIVE_REGION_NOT_WORK_REGION','REG_CITY_NOT_LIVE_CITY',
        'REG_CITY_NOT_WORK_CITY','LIVE_CITY_NOT_WORK_CITY'
    ]
    df['MOBILITY_SCORE'] = df[mobility_cols].sum(axis=1)
    df.drop(columns=mobility_cols, inplace=True, errors='ignore')


# ===================================================
# === APPLY ALL FEATURE ENGINEERING FUNCTIONS ======
# ===================================================

def run_feature_engineering(applications):

    Total_contact_point(applications)
    Count_doc_submitted(applications)
    Ext_Source(applications)
    Social_circle(applications)
    credit_bureau_risk(applications)
    property_area(applications)
    categorical_simplification(applications)
    regional_risk_score(applications)
    mobility_score(applications)

    return applications


# Apply the pipeline
applications = run_feature_engineering(applications)

# Final shape
applications.shape


(36102, 38)

In [679]:
applications.columns

Index(['TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR',
       'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'NAME_TYPE_SUITE',
       'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'OWN_CAR_AGE',
       'OCCUPATION_TYPE', 'REGION_RATING_CLIENT_W_CITY',
       'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START',
       'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE',
       'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE', 'AGE_AT_LOAN',
       'JOB_AGE_AT_LOAN', 'DOCS_CHANGE_AT', 'YEARS_LAST_PHONE_CHANGE',
       'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_GOODS_RATIO',
       'INCOME_PER_PERSON', 'TOTAL_CONTACT_POINTS', 'COUNT_DOCS',
       'EXT_SOURCE_MEAN', 'SOCIAL_RISK_RATIO', 'TOTAL_BUREAU_REQUESTS',
       'PROPERTY_AREA', 'EDU_SIMPLIFIED', 'INCOME_SIMPLIFIED', 'FAMILY_TYPE',
       'REGION_RATING_DIFF', 'MOBILITY_SCORE'],
      dtype='object')

In [680]:
applications['DOCS_CHANGE_AT']

0         6
1         1
2         7
3         7
4         9
         ..
36097     8
36098     2
36099    14
36100     1
36101    11
Name: DOCS_CHANGE_AT, Length: 36102, dtype: int64

# **Step 4: Binning**

In this step, we transformed continuous numerical variables into categorical groups to improve interpretability, reduce noise, and help the model capture non-linear patterns in the data.  
Binning is especially useful when raw numeric values do not carry clear meaning on their own but their ranges or groups do.

---

## **4.1 Binning Children Count (CNT_CHILDREN)**

**What We Did:**  

We converted the exact number of children into meaningful family-size categories:

| Original Value | New Category          |
| -------------- | --------------------- |
| 0              | No Children           |
| 1–3            | Small Family (1–3)    |
| 3–5            | Medium Family (3–5)   |
| 5+             | Large Family (5+)     |

**Why We Did It:**  

- The financial strain of raising children does not increase linearly with each additional child.  
- Models struggle when the numeric value has low variance but high impact.  
- Categorizing helps capture real-world segments such as “large families” or “no children.”  
- Improves interpretability for business understanding and reporting.  
- This turns a noisy numeric variable into a clean, business-meaningful indicator of household responsibility.

---

## **4.2 Binning Population Density (REGION_POPULATION_RELATIVE)**

**What We Did:**  

We grouped region population density using quartiles into:

- Very Low Density  
- Low Density  
- Medium Density  
- High Density  

These bins ensure balanced distribution because they are based on the actual dataset's quartiles (Q1, Q2, Q3).

**Why We Did It:**  

- Population density reflects urbanization level, which affects stability, job access, and credit behavior.  
- Raw values are small decimals (0–1), often skewed and hard for models to interpret.  
- Binning removes noise and allows the model to distinguish:  
  - Rural areas  
  - Semi-urban areas  
  - Urban areas  
  - High-density metro regions  
- This enhances the model’s ability to detect region-based default risk patterns.


In [681]:
def child_count(df):
    df['CNT_CHILDREN'] = pd.cut(
        df['CNT_CHILDREN'],
        bins=[-1, 0, 3, 5, df['CNT_CHILDREN'].max()],
        labels=[
            'No Children',
            'Small Family (1–3)',
            'Medium Family (3–5)',
            'Large Family (5+)'
        ],
        include_lowest=True
    )

def population_density_category(df):
    q1 = df['REGION_POPULATION_RELATIVE'].quantile(0.25)
    q2 = df['REGION_POPULATION_RELATIVE'].quantile(0.50)
    q3 = df['REGION_POPULATION_RELATIVE'].quantile(0.75)

    df['POP_DENSITY_CAT'] = pd.cut(
        df['REGION_POPULATION_RELATIVE'],
        bins=[0, q1, q2, q3, 1],
        labels=['Very Low', 'Low', 'Medium', 'High'],
        include_lowest=True
    )
    df.drop(columns='REGION_POPULATION_RELATIVE', inplace=True, errors='ignore')
child_count(applications)
population_density_category(applications)


In [682]:
applications['POP_DENSITY_CAT'].value_counts()

POP_DENSITY_CAT
Medium      9522
Low         9422
Very Low    9217
High        7941
Name: count, dtype: int64

In [683]:
applications.columns

Index(['TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR',
       'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'NAME_TYPE_SUITE',
       'NAME_HOUSING_TYPE', 'OWN_CAR_AGE', 'OCCUPATION_TYPE',
       'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START',
       'HOUR_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE',
       'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE',
       'AGE_AT_LOAN', 'JOB_AGE_AT_LOAN', 'DOCS_CHANGE_AT',
       'YEARS_LAST_PHONE_CHANGE', 'CREDIT_INCOME_RATIO',
       'ANNUITY_INCOME_RATIO', 'CREDIT_GOODS_RATIO', 'INCOME_PER_PERSON',
       'TOTAL_CONTACT_POINTS', 'COUNT_DOCS', 'EXT_SOURCE_MEAN',
       'SOCIAL_RISK_RATIO', 'TOTAL_BUREAU_REQUESTS', 'PROPERTY_AREA',
       'EDU_SIMPLIFIED', 'INCOME_SIMPLIFIED', 'FAMILY_TYPE',
       'REGION_RATING_DIFF', 'MOBILITY_SCORE', 'POP_DENSITY_CAT'],
      dtype='object')

In [684]:
from pathlib import Path

# Get the current notebook's directory
CURRENT_NOTEBOOK_DIR = Path().resolve()

# Assume project root is one level up from notebooks/
BASE_DIR = CURRENT_NOTEBOOK_DIR.parent.parent
Processed_Data_Path = BASE_DIR / "data"/ "processed"/ "Feature-Engineered"
Processed_Data_Path

WindowsPath('D:/Repos/Data_viz/technerds/data/processed/Feature-Engineered')

In [685]:
applications.to_csv(Processed_Data_Path/"application_data.csv")